# CHSM8, nano

The whole pipeline on a laptop CPU in about 15 minutes, with **the same scripts our real runs used** (`src/`), on one month of
Lichess and a 1M-parameter model. Only the sizes differ from `recipes/`: depth 4, width 128, fp32, a minute per stage.

Stages: pretraining data → pretraining → post-training data → SFT → DPO → games-only continuation → a player model → Elo.
The labeler stages need a GPU (vLLM) and are in `full.ipynb`.

Needs `pip install -e .` and `huggingface-cli login` with access to [EmbeddingGemma-300m](https://huggingface.co/google/embeddinggemma-300m) (SFT and DPO).

In [ ]:
import os, subprocess, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
LIMIT = 3000        # pretraining games; raise it for a better nano model
MONTH = "2014-01"   # one Lichess month for everything
PY = sys.executable
NANO = "--architecture llama --llama-qk-norm --depth 4 --d-model 128 --n-heads 4 --d-ff 512 --dropout 0.0 --no-use-flash-attn --dtype fp32 --legal-penalty-weight 0.5 --batch-size 8"      # the nano model; recipes/ use --depth 12 --d-model 960 --n-heads 15 --d-ff 3520, bf16, flash-attn
LAB = "gemma-4-31b,gemma-4-31b-patched,gemma-3-1b-distill,gemma-3-1b-distill-patched,gemma-3-1b-distill-clean"
os.environ.update(OMP_NUM_THREADS="8", TOKENIZERS_PARALLELISM="false")

def run(cmd):
    print("$", cmd)
    subprocess.run("set -o pipefail; " + cmd, shell=True, check=True, executable="/bin/bash")

## 1. Pretraining data
`LIMIT` games of `MONTH` from [chsm8-pre-training](https://huggingface.co/datasets/LegumMagister/chsm8-pre-training), then the
dataset's own `prepare.sh`: parquet → tier 2b (move codes) → factorized rows `(kind, src, dst, piece, promo)` → packed
records of 513 rows → shuffled. The same command as `recipes/0_data.sh`, on all 162 months.

In [ ]:
from huggingface_hub import snapshot_download
import pyarrow.dataset as ds, pyarrow.parquet as pq
PRE = snapshot_download("LegumMagister/chsm8-pre-training", repo_type="dataset", allow_patterns=[f"data/{MONTH}/*"])
os.makedirs(f"data/nano/hf/{MONTH}", exist_ok=True)
pq.write_table(ds.dataset(f"{PRE}/data/{MONTH}").to_table().slice(0, LIMIT), f"data/nano/hf/{MONTH}/part-00000.parquet")
run(f"SRC=data/nano/hf WORK=data/nano/pre BUCKETS=4 WORKERS=4 PYTHON={PY} bash prep/pretraining/prepare.sh | tail -3")

## 2. Pretraining
`src/train_pretrain.py`, as in `recipes/1_pretrain.sh` (there: 78.3B positions on 4 GPUs, about 40 hours).

In [ ]:
run(f"{PY} src/train_pretrain.py --prepacked-dir data/nano/pre/3p-shuffled --prepacked-without-replacement --prepacked-sequential "
    f"{NANO} --use-cross-attention --fast-dummy-cross-attn --lr 2e-3 --weight-decay 0.1 --warmup-ratio .05 --time-budget 60 "
    f"--checkpoint-dir runs/nano/pt --checkpoint-every 1e9 --eval-every 1e9 --eval-steps 5 --log-every 50 "
    f"--run-name nano_pt --results-dir runs/nano | tail -3")

## 3. Post-training data
The described sides of `MONTH` from [chsm8-post-training](https://huggingface.co/datasets/LegumMagister/chsm8-post-training)
(first file of each split), joined with their moves; then our two builders: the SFT/DPO corpus and the packed games for
the games-only continuation. Top players: [chsm8-top-players](https://huggingface.co/datasets/LegumMagister/chsm8-top-players)
→ one player's pack (Fischer) and the player-prompt corpus for SFT.

In [ ]:
run(f"{PY} prep/posttraining.py --first-files --months {MONTH} --pre {PRE}")
run(f"{PY} src/build_sft_corpus.py data/chsm8-post-training-unlabeled/data --out data/nano/sft --workers 4 | tail -1")
run(f"{PY} src/build_games_packed.py --source pool --out data/nano/games --workers 4 | tail -1")
run(f"{PY} prep/players.py | tail -2")
os.makedirs("data/openings", exist_ok=True)
for f in "abcde":
    run(f"curl -sL https://raw.githubusercontent.com/lichess-org/chess-openings/master/{f}.tsv -o data/openings/{f}.tsv")
run(f"{PY} src/build_games_packed.py --source players --inputs data/top_players/games/fischer.jsonl --out data/nano/players/fischer --holdout 2000 --repeat-to 20000 | tail -1")
run(f"{PY} src/build_player_corpus.py --out data/nano/sft_players --workers 4 | tail -1")

## 4. SFT
`src/sft_ablate.py`: cross-attention to the frozen EmbeddingGemma-300m in the top layers, 10% label dropout, the pinned
no-text term, player prompts. As in `recipes/7_sft.sh` (there: 6 of 12 layers, 27.1M sides, 8 GPUs, 3.1 hours).

In [ ]:
run(f"{PY} src/sft_ablate.py --trial nano_sft --results runs/nano/sft.jsonl --data data/nano/sft --checkpoint runs/nano/pt/latest.pt "
    f"--encoder google/embeddinggemma-300m --encoder-kind token --pool mean --meta-labels '' --labelers {LAB} --eval-labelers {LAB} "
    f"--cond-layers 2 --lr 5e-7 --cond-lr 8e-4 --final-lr-frac 0.0 --labels-per-step 0 --games 8 --token-budget 4096 "
    f"--cfg-dropout 0.1 --freeze-dummy --player-data data/nano/sft_players --player-epochs 3 --player-peak 1 --planned-games -1 "
    f"--time-budget 60 --val-every 2000 --val-games 16 --eval-games 16 --eval-test --save --save-dir runs/nano/sft --ckpt-every 600 | tail -2 | cut -c1-300")

## 5. DPO
`src/dpo_sft.py`: the chosen sample is a game under its own description, the rejected one the same game under another
game's description. As in `recipes/8_dpo.sh` (there: 20 minutes on 1 GPU).

In [ ]:
run(f"{PY} src/dpo_sft.py --trial nano_dpo --pair-mode label --beta 0.5 --nll-weight 1.0 --kl-weight 1.0 --freeze-dummy "
    f"--init runs/nano/sft/nano_sft.pt --data data/nano/sft --label-emb= --labelers {LAB} --time-budget 30 --games 8 --token-budget 4096 "
    f"--eval-every 1000000 --eval-games 16 --save-dir runs/nano/dpo --results runs/nano/dpo.jsonl | tail -2 | cut -c1-300")

## 6. Games-only continuation and a player model
`src/train_pretrain.py` again: continue the pretrained model on the post-training games without text (chsm8-player-base),
then fine-tune it on one player's own moves (`--loss-side focal`), and measure how often it plays that player's move.
As in `recipes/2_player_base.sh` and `recipes/3_players.sh`.

In [ ]:
run(f"{PY} src/train_pretrain.py --prepacked-dir data/nano/games --init-checkpoint runs/nano/pt/latest.pt {NANO} --weight-decay 0.1 "
    f"--warmup-ratio 0.05 --lr-schedule cosine --cosine-min-lr-ratio 0.0 --time-budget 20 --eval-every-frac 0.1 --eval-fixed --eval-steps 5 "
    f"--log-every 50 --checkpoint-every 1e9 --run-name nano_base --checkpoint-dir runs/nano/base --results-dir runs/nano "
    f"--no-use-cross-attention --xattn-mode drop --lr 5e-05 --milestone-checkpoint-percent 0 --milestone-checkpoint-extra-percent 0 | tail -1")
import json
d = json.load(open("data/nano/players/fischer/summary.json"))
POS = int(2 * d["unique_train_games"] * d["tokens_train"] / d["train"]["games"])
run(f"{PY} src/train_pretrain.py {NANO} --no-use-cross-attention --xattn-mode drop --lr 3e-5 --warmup-ratio 0.05 --lr-schedule cosine "
    f"--cosine-min-lr-ratio 0.0 --lr-schedule-by-positions --max-train-positions {POS} --time-budget 60 --eval-every-frac 0.1 --eval-fixed "
    f"--eval-steps 5 --log-every 50 --checkpoint-every 100000 --loss-side focal --milestone-checkpoint-percent 0 "
    f"--milestone-checkpoint-extra-percent 0 --results-dir runs/nano --prepacked-dir data/nano/players/fischer "
    f"--init-checkpoint runs/nano/base/latest.pt --run-name nano_fischer --checkpoint-dir runs/nano/fischer | tail -1")
run(f"{PY} scripts/eval_player_fit.py runs/nano/fischer/latest.pt data/nano/players/fischer --split test --tag nano_fischer | tail -1")

## 7. Elo
Games against Stockfish 17.1 at `UCI_Elo` 2000, 0.05 s per move: `src/benchmark_play.py` for games-only models,
`src/eval_sft.py` for text models (with a prompt). Set `STOCKFISH` to your binary. As in `recipes/9_eval.sh` (1,000 games).

In [ ]:
SF = os.environ.get("STOCKFISH", "stockfish")
run(f"{PY} src/benchmark_play.py runs/nano/pt/latest.pt --engine {SF} --games 4 --engine-elo 2000 --engine-time 0.05 "
    f"--max-plies 240 --parallel-games 4 --out runs/nano/elo/pt/shard0 > /dev/null")
run(f"{PY} src/summarize_elo1k.py runs/nano/elo/pt | tail -1")
run(f"{PY} src/eval_sft.py --prompt 'attack the king at all costs' runs/nano/dpo/nano_dpo.pt --engine {SF} --games 4 "
    f"--engine-elo 2000 --engine-time 0.05 --max-plies 240 --parallel-games 4 --out runs/nano/elo/dpo/part0 > /dev/null")